<a href="https://colab.research.google.com/github/HARSITHRAM/Interactive-Online-Class/blob/main/int_class03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install paho-mqtt deepface
!wget https://github.com/googlefonts/roboto/raw/main/src/hinted/Roboto-Bold.ttf -O /content/bold.ttf
!apt-get update
!apt-get install -y libfreetype6-dev libpng-dev
!pip install -U Pillow

--2025-04-11 14:28:03--  https://github.com/googlefonts/roboto/raw/main/src/hinted/Roboto-Bold.ttf
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/googlefonts/roboto-2/raw/main/src/hinted/Roboto-Bold.ttf [following]
--2025-04-11 14:28:03--  https://github.com/googlefonts/roboto-2/raw/main/src/hinted/Roboto-Bold.ttf
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/googlefonts/roboto-2/main/src/hinted/Roboto-Bold.ttf [following]
--2025-04-11 14:28:03--  https://raw.githubusercontent.com/googlefonts/roboto-2/main/src/hinted/Roboto-Bold.ttf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.

In [ ]:
from google.colab.patches import cv2_imshow
from IPython.display import display, clear_output
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
from PIL import Image
from deepface import DeepFace
import paho.mqtt.client as mqtt
import socket
import time
import re

# Convert JavaScript object to OpenCV image
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    return cv2.imdecode(jpg_as_np, flags=1)

# Overlay emotion text on image
def overlay_emotion_text(frame, emotion, position=(50, 50)):
    cv2.putText(frame, emotion, position, cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)

# Get a single video frame from webcam
def video_frame():
    js = eval_js('''
        async function captureImage() {
            const video = document.createElement('video');
            const canvas = document.createElement('canvas');
            const ctx = canvas.getContext('2d');
            const stream = await navigator.mediaDevices.getUserMedia({ video: true });
            video.srcObject = stream;
            await video.play();
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
            stream.getTracks().forEach(t => t.stop());
            return canvas.toDataURL('image/jpeg');
        }
        captureImage();
    ''')
    return js

# MQTT setup
broker_address = "b7567dff6628421e8bfca997634b396b.s1.eu.hivemq.cloud"
broker_port = 8883
topic = "teacher_input"
status_topic = "student_status"

client = mqtt.Client(client_id="colab_publisher", protocol=mqtt.MQTTv311)
client.tls_set()
client.username_pw_set("HARSITHRAM", "Ram@2005")

# Global variables
student_name = "student_01"
latest_emotion = None
active_emotions = ["happy", "neutral"]
inactive_emotions = ["sad", "angry", "disgust", "fear", "surprise"]
teacher_request = ""
send_active_status = False

# NLP to detect student name from teacher's message
def extract_student_name(message):
    match = re.search(r'(student_\d+)', message.lower())
    return match.group(1) if match else None

# Callback when a message is received
def on_message(client, userdata, msg):
    global send_active_status, teacher_request
    message = msg.payload.decode()
    print(f"\n\U0001F4E9 Received message from teacher: {message}")
    teacher_request = message
    name = extract_student_name(message)
    if name == student_name:
        send_active_status = True
    else:
        print("\u2139\ufe0f No matching student name found.")

client.on_message = on_message

try:
    client.connect(broker_address, broker_port)
    client.subscribe(topic)
    client.loop_start()
    print("Connected and subscribed to MQTT broker")
except Exception as e:
    print(f"MQTT Connection error: {e}")

try:
    while True:
        # Capture frame
        js_reply = video_frame()
        frame = js_to_image(js_reply)

        try:
            result = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)
            emotion = result[0]['dominant_emotion']
            latest_emotion = emotion
            overlay_emotion_text(frame, emotion)

            # Display frame
            clear_output(wait=True)
            display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

            # Always send message if student is inactive
            if emotion in inactive_emotions:
                client.publish(status_topic, f"{student_name} is not active")

            # Send active status only if teacher asked
            if send_active_status:
                msg = f"{student_name} is active" if emotion in active_emotions else f"{student_name} is not active"
                client.publish(status_topic, msg)
                send_active_status = False  # Reset trigger after response

        except Exception as e:
            print(f"Emotion analysis failed: {e}")

        time.sleep(2)

except KeyboardInterrupt:
    print("\n\u274C Interrupted by user.")
finally:
    client.loop_stop()
    cv2.destroyAllWindows()
    print("Process finished.")


✅ Connected with result code 0
